# 03 - English Speech Synthesis with Voice Cloning (F5-TTS)

This notebook uses **F5-TTS** to generate English speech from translated segments,
cloning the original speaker's voice for a natural result.

**Requirements:** Google Colab with GPU runtime (T4 or better).

**Input:** Translation JSON files + voice reference WAV files from Google Drive.

**Output:** Per-segment WAV files saved to Google Drive.

In [ ]:
# Install F5-TTS
!pip install -q f5-tts soundfile numpy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/AIVideoLanguageTransformation'
TRANSLATIONS_DIR = os.path.join(PROJECT_DIR, 'data', 'translations')
VOICE_REF_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'voice_reference')
TTS_DIR = os.path.join(PROJECT_DIR, 'data', 'tts')

print(f'Translations: {os.listdir(TRANSLATIONS_DIR) if os.path.exists(TRANSLATIONS_DIR) else "NOT FOUND"}')
print(f'Voice refs: {os.listdir(VOICE_REF_DIR) if os.path.exists(VOICE_REF_DIR) else "NOT FOUND"}')

In [ ]:
from f5_tts.api import F5TTS
import json
import soundfile as sf
import numpy as np
from tqdm import tqdm

# Initialize F5-TTS
print('Loading F5-TTS model...')
tts = F5TTS()
print('Model loaded.')

In [ ]:
def load_voice_refs(voice_ref_dir):
    """Load available voice reference files, mapped by speaker label."""
    refs = {}
    for f in os.listdir(voice_ref_dir):
        if not f.endswith('.wav'):
            continue
        # Match files like SPEAKER_00_ref.wav, male_ref.wav, etc.
        key = f.replace('_ref.wav', '').replace('_ref', '')
        refs[key] = os.path.join(voice_ref_dir, f)
    print(f'Voice references loaded: {refs}')
    return refs


def get_voice_ref(seg, voice_refs, default_ref):
    """Get the appropriate voice reference for a segment based on speaker label."""
    speaker = seg.get('speaker', '')
    # Try exact match first (e.g., SPEAKER_00)
    if speaker in voice_refs:
        return voice_refs[speaker]
    # Try partial match (e.g., speaker label contains 'male' or 'female')
    for key, path in voice_refs.items():
        if key.lower() in speaker.lower() or speaker.lower() in key.lower():
            return path
    return default_ref


def synthesize_segments(translation_path, voice_refs, default_ref, output_dir):
    """Synthesize English speech for all segments, using per-speaker voice references."""
    os.makedirs(output_dir, exist_ok=True)

    with open(translation_path, 'r', encoding='utf-8') as f:
        segments = json.load(f)

    print(f'Synthesizing {len(segments)} segments...')

    for seg in tqdm(segments):
        text = seg.get('text_en', seg.get('text_en_deepl', ''))
        if not text.strip():
            continue

        output_path = os.path.join(output_dir, f"segment_{seg['id']:04d}.wav")

        # Skip if already synthesized (for resumability)
        if os.path.exists(output_path):
            audio_data, sr = sf.read(output_path)
            seg['tts_file'] = os.path.basename(output_path)
            seg['tts_duration'] = round(len(audio_data) / sr, 3)
            continue

        ref_path = get_voice_ref(seg, voice_refs, default_ref)

        try:
            wav, sr, _ = tts.infer(
                ref_file=ref_path,
                ref_text="",  # Empty for zero-shot cloning
                gen_text=text,
                speed=1.0,
            )

            sf.write(output_path, wav, sr)
            seg['tts_file'] = os.path.basename(output_path)
            seg['tts_duration'] = round(len(wav) / sr, 3)

        except Exception as e:
            print(f"  Error on segment {seg['id']}: {e}")
            seg['tts_file'] = None
            seg['tts_duration'] = 0

    # Save updated segments with TTS metadata
    with open(translation_path, 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)

    return segments

In [ ]:
# Load per-speaker voice references
voice_refs = load_voice_refs(VOICE_REF_DIR)
voice_ref_files = sorted([f for f in os.listdir(VOICE_REF_DIR) if f.endswith('.wav')])
default_ref = os.path.join(VOICE_REF_DIR, voice_ref_files[0]) if voice_ref_files else None

translation_files = sorted([f for f in os.listdir(TRANSLATIONS_DIR) if f.endswith('_en.json')])
print(f'Translation files: {translation_files}')

# Delete old TTS segments to force re-synthesis with new voice references
import shutil
for trans_file in translation_files:
    stem = trans_file.replace('_en.json', '')
    old_dir = os.path.join(TTS_DIR, f'{stem}_segments')
    if os.path.exists(old_dir):
        print(f'Removing old segments: {old_dir}')
        shutil.rmtree(old_dir)

for trans_file in translation_files:
    stem = trans_file.replace('_en.json', '')
    trans_path = os.path.join(TRANSLATIONS_DIR, trans_file)
    output_dir = os.path.join(TTS_DIR, f'{stem}_segments')

    print(f'\n{"="*60}')
    print(f'Video: {stem}')
    synthesize_segments(trans_path, voice_refs, default_ref, output_dir)

print('\nAll synthesis complete!')

In [ ]:
# Listen to a sample segment
from IPython.display import Audio
import os

# Pick the first video's first segment
sample_dir = os.path.join(TTS_DIR, os.listdir(TTS_DIR)[0])
sample_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.wav')])
if sample_files:
    sample_path = os.path.join(sample_dir, sample_files[0])
    print(f'Playing: {sample_path}')
    Audio(sample_path)